# 🏥 Chatbot RAG — Bupa Chile
###integrante:Diego Yañez
### ISY0101 Ingenieria de Soluciones con IA — Evaluacion Parcial N°1

- Embeddings: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- LLM: `google/flan-t5-large`
- Vector Store: `ChromaDB`
- Framework: `LangChain (API moderna LCEL)`



## Paso 1 — Instalacion

In [1]:
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-core langchain-text-splitters chromadb sentence-transformers transformers torch accelerate
print('OK instalacion completa')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

## Paso 2 — Documentos Bupa Chile

In [2]:
from langchain_core.documents import Document

documentos_bupa = [
    Document(
        page_content="COBERTURAS PLAN BUPA ESENCIAL 2024. El plan Bupa Esencial cubre hasta el 70% de los gastos en consultas medicas con medicos de la red Bupa. Las consultas con especialistas tienen cobertura del 60% hasta un tope de 45000 pesos por consulta. Los examenes de laboratorio tienen cobertura del 80% en laboratorios de la red. Hospitalizacion: cobertura del 75% hasta un tope de 3500000 pesos por evento. Las urgencias estan cubiertas al 90% en cualquier clinica de la red Bupa. No cubre: medicina alternativa, cirugias esteticas, tratamientos experimentales.",
        metadata={"fuente": "Cartilla_Coberturas_Esencial_2024.pdf", "pagina": 3}
    ),
    Document(
        page_content="COBERTURAS PLAN BUPA PREMIUM 2024. El plan Bupa Premium cubre el 90% de todos los gastos medicos en la red Bupa. Consultas medicas generales: cobertura 90% sin tope. Especialistas: cobertura 85% sin tope. Examenes e imagenes: cobertura 90%. Hospitalizacion: cobertura 90% hasta 8000000 pesos por evento. Urgencias: cobertura 95% en cualquier clinica del pais. Incluye cobertura dental basica 50%. Incluye telemedicina ilimitada sin costo adicional.",
        metadata={"fuente": "Cartilla_Coberturas_Premium_2024.pdf", "pagina": 2}
    ),
    Document(
        page_content="PROCESO DE REEMBOLSO BUPA CHILE. Para solicitar un reembolso el afiliado debe: 1. Completar el formulario de reembolso en www.bupa.cl o en cualquier sucursal. 2. Adjuntar boleta o factura original. 3. Adjuntar detalle de prestaciones. 4. Adjuntar fotocopia de cedula de identidad. 5. En caso de hospitalizacion adjuntar epicrisis. El plazo es de 90 dias desde la prestacion. El procesamiento tarda 10 dias habiles. El reembolso se deposita en la cuenta bancaria del afiliado.",
        metadata={"fuente": "Reglamento_Reembolsos_2024.pdf", "pagina": 7}
    ),
    Document(
        page_content="HORARIOS CENTROS MEDICOS BUPA REGION METROPOLITANA. Centro Medico Providencia: Av. Providencia 1234. Lunes a Viernes 8:00-20:00, Sabado 9:00-14:00. Centro Medico Las Condes: Av. Apoquindo 5678. Lunes a Viernes 7:30-21:00, Sabado 9:00-15:00. Centro Medico Maipu: Av. Pajaritos 890. Lunes a Viernes 8:00-19:00, Sabado 9:00-13:00. Clinica Bupa Santiago urgencias 24 horas: Av. Vitacura 5951. Para agendar hora: llamar al 600 360 0000 o www.bupa.cl.",
        metadata={"fuente": "Directorio_Centros_RM_2024.pdf", "pagina": 1}
    ),
    Document(
        page_content="COMO AGENDAR HORA MEDICA EN BUPA CHILE. Formas de agendar: 1. Portal web www.bupa.cl seccion Agenda tu hora. 2. Aplicacion movil Bupa Chile en App Store o Google Play. 3. Call center 600 360 0000 Lunes a Viernes 8:00-20:00 y Sabados 9:00-14:00. Para cancelar o modificar una hora debes hacerlo con al menos 24 horas de anticipacion. Cancelaciones con menos de 24 horas generan cargo de 5000 pesos. Las urgencias no requieren agendamiento previo.",
        metadata={"fuente": "Guia_Agendamiento_2024.pdf", "pagina": 1}
    ),
    Document(
        page_content="PROTOCOLO EXAMENES DE SANGRE HEMOGRAMA Y PERFIL BIOQUIMICO. Para hemograma perfil lipidico glicemia y funcion hepatica: ayuno minimo de 8 horas, se recomienda 10 a 12 horas. Solo se permite tomar agua durante el ayuno. No consumir alcohol 48 horas antes. Informar sobre medicamentos al tecnico. Llegar con cedula de identidad y orden medica. Horario de toma de muestras: Lunes a Viernes 7:30 a 11:00. Resultados en 24 a 48 horas en www.bupa.cl seccion Mis examenes.",
        metadata={"fuente": "Protocolo_Examenes_Laboratorio.pdf", "pagina": 4}
    ),
    Document(
        page_content="TELEMEDICINA BUPA CHILE. Consultas medicas por videollamada disponibles. Especialidades: Medicina General, Pediatria, Psicologia, Nutricion, Dermatologia. Disponibilidad: Lunes a Domingo 8:00 a 22:00 horas. Costo Plan Esencial: 15000 pesos con bonificacion 50%. Costo Plan Premium: sin costo incluido en el plan. Como acceder: App Bupa Chile seccion Telemedicina. Requisito: internet estable, camara y microfono funcionando.",
        metadata={"fuente": "Guia_Telemedicina_2024.pdf", "pagina": 1}
    ),
]

print(f'OK {len(documentos_bupa)} documentos cargados')

OK 7 documentos cargados


## Paso 3 — Embeddings + ChromaDB

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = splitter.split_documents(documentos_bupa)
print(f'OK chunking: {len(chunks)} fragmentos')

# Embeddings gratuitos en español
print('Cargando modelo de embeddings (1 minuto aprox)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='bupa_kb'
)

retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})
print('OK ChromaDB y retriever listos')

OK chunking: 8 fragmentos
Cargando modelo de embeddings (1 minuto aprox)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

OK ChromaDB y retriever listos


## Paso 4 — Cargar LLM

In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from langchain_huggingface import HuggingFacePipeline
import torch

MODEL = 'google/flan-t5-large'
print(f'Cargando {MODEL}...')

tokenizer = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL)

# Funcion personalizada que reemplaza el pipeline roto
def run_flan(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Wrapper compatible con LangChain
from langchain_core.language_models import BaseLLM
from langchain_core.outputs import LLMResult, Generation
from typing import List, Optional, Any

class FlanLLM(BaseLLM):
    def _generate(self, prompts: List[str], stop: Optional[List[str]] = None, **kwargs: Any) -> LLMResult:
        generations = []
        for p in prompts:
            text = run_flan(p)
            generations.append([Generation(text=text)])
        return LLMResult(generations=generations)

    @property
    def _llm_type(self) -> str:
        return 'flan-t5'

llm = FlanLLM()
print('OK modelo listo')

Cargando google/flan-t5-large...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


OK modelo listo


## Paso 5 — Pipeline RAG (API Moderna LCEL)

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

PROMPT_TEMPLATE = """Eres un asistente de salud de Bupa Chile.
Responde usando SOLO la informacion del contexto.
Si no tienes la informacion di: No tengo esa informacion, llama al 600 360 0000.
Nunca des diagnosticos medicos. Responde en español.

Contexto: {context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template=PROMPT_TEMPLATE
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('OK pipeline RAG construido')

OK pipeline RAG construido


## Paso 6 — Funcion de Consulta

In [10]:
def consultar_bupa(pregunta: str):
    palabras_urgencia = [
        'emergencia', 'urgencia', 'dolor pecho', 'accidente',
        'sangrado', 'desmayo', 'no puedo respirar', 'infarto'
    ]
    if any(p in pregunta.lower() for p in palabras_urgencia):
        print("=" * 60)
        print("⚠️  URGENCIA — Llama al 131 o ve a Clinica Bupa Santiago")
        print("   Av. Vitacura 5951 — Urgencias 24/7")
        print("=" * 60)
        return

    print(f"👤 Paciente: {pregunta}")
    print("-" * 60)

    # Recuperar fragmentos relevantes
    docs = retriever.invoke(pregunta)

    if not docs:
        print("🤖 Asistente Bupa: No tengo esa informacion. Llama al 600 360 0000.")
    else:
        # Usar el fragmento mas relevante como respuesta directa
        mejor_doc = docs[0]
        print(f"🤖 Asistente Bupa:\n{mejor_doc.page_content}")
        print(f"\n📄 Fuente: {mejor_doc.metadata.get('fuente')} — pág. {mejor_doc.metadata.get('pagina')}")

    print("=" * 60)

print('OK funcion lista')

OK funcion lista


## Paso 7 — Pruebas

In [11]:
consultar_bupa("Que cubre el plan Bupa Esencial para consultas con especialistas?")

👤 Paciente: Que cubre el plan Bupa Esencial para consultas con especialistas?
------------------------------------------------------------
🤖 Asistente Bupa:
COBERTURAS PLAN BUPA ESENCIAL 2024. El plan Bupa Esencial cubre hasta el 70% de los gastos en consultas medicas con medicos de la red Bupa. Las consultas con especialistas tienen cobertura del 60% hasta un tope de 45000 pesos por consulta. Los examenes de laboratorio tienen cobertura del 80% en laboratorios de la red. Hospitalizacion: cobertura del 75% hasta un tope de 3500000 pesos por evento. Las urgencias estan cubiertas al 90% en cualquier clinica de la red Bupa. No cubre: medicina alternativa, cirugias

📄 Fuente: Cartilla_Coberturas_Esencial_2024.pdf — pág. 3


In [12]:
consultar_bupa("Como puedo pedir un reembolso?")

👤 Paciente: Como puedo pedir un reembolso?
------------------------------------------------------------
🤖 Asistente Bupa:
PROCESO DE REEMBOLSO BUPA CHILE. Para solicitar un reembolso el afiliado debe: 1. Completar el formulario de reembolso en www.bupa.cl o en cualquier sucursal. 2. Adjuntar boleta o factura original. 3. Adjuntar detalle de prestaciones. 4. Adjuntar fotocopia de cedula de identidad. 5. En caso de hospitalizacion adjuntar epicrisis. El plazo es de 90 dias desde la prestacion. El procesamiento tarda 10 dias habiles. El reembolso se deposita en la cuenta bancaria del afiliado.

📄 Fuente: Reglamento_Reembolsos_2024.pdf — pág. 7


In [13]:
consultar_bupa("Cuantas horas de ayuno necesito para un hemograma?")

👤 Paciente: Cuantas horas de ayuno necesito para un hemograma?
------------------------------------------------------------
🤖 Asistente Bupa:
PROTOCOLO EXAMENES DE SANGRE HEMOGRAMA Y PERFIL BIOQUIMICO. Para hemograma perfil lipidico glicemia y funcion hepatica: ayuno minimo de 8 horas, se recomienda 10 a 12 horas. Solo se permite tomar agua durante el ayuno. No consumir alcohol 48 horas antes. Informar sobre medicamentos al tecnico. Llegar con cedula de identidad y orden medica. Horario de toma de muestras: Lunes a Viernes 7:30 a 11:00. Resultados en 24 a 48 horas en www.bupa.cl seccion Mis examenes.

📄 Fuente: Protocolo_Examenes_Laboratorio.pdf — pág. 4


In [14]:
consultar_bupa("Cuales son los horarios del centro medico de Las Condes?")

👤 Paciente: Cuales son los horarios del centro medico de Las Condes?
------------------------------------------------------------
🤖 Asistente Bupa:
HORARIOS CENTROS MEDICOS BUPA REGION METROPOLITANA. Centro Medico Providencia: Av. Providencia 1234. Lunes a Viernes 8:00-20:00, Sabado 9:00-14:00. Centro Medico Las Condes: Av. Apoquindo 5678. Lunes a Viernes 7:30-21:00, Sabado 9:00-15:00. Centro Medico Maipu: Av. Pajaritos 890. Lunes a Viernes 8:00-19:00, Sabado 9:00-13:00. Clinica Bupa Santiago urgencias 24 horas: Av. Vitacura 5951. Para agendar hora: llamar al 600 360 0000 o www.bupa.cl.

📄 Fuente: Directorio_Centros_RM_2024.pdf — pág. 1


In [15]:
consultar_bupa("Tengo un dolor pecho muy fuerte que hago?")

⚠️  URGENCIA — Llama al 131 o ve a Clinica Bupa Santiago
   Av. Vitacura 5951 — Urgencias 24/7


## Paso 8 — Chat Interactivo

In [16]:
print("CHATBOT BUPA CHILE — escribe 'salir' para terminar")
print("=" * 60)
while True:
    pregunta = input("Tu consulta: ").strip()
    if not pregunta:
        continue
    if pregunta.lower() == 'salir':
        print("Sesion finalizada.")
        break
    consultar_bupa(pregunta)

CHATBOT BUPA CHILE — escribe 'salir' para terminar
Tu consulta: salir
Sesion finalizada.
